# Laboratorio 17. Atención

**Pregunta guía:** ¿Cómo decide un mecanismo de atención qué posiciones consultar?

## Objetivos
Al finalizar podrás calcular queries, keys y values; aplicar escala y softmax; visualizar pesos de atención y máscara causal.

## Prerrequisitos y conexión
LSTM/GRU (16) procesan recurrentemente; la atención establece interacciones entre posiciones. Se requiere Python básico; cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
Scaled dot-product attention: $Attention(Q,K,V)=softmax(QK^T/\sqrt{d_k})V$. Una máscara causal impide consultar posiciones futuras.

## Problema y datos
Un lote pequeño contiene vectores por posición y relaciones de atención interpretables. Todos los datos se generan en el notebook: no se requieren descargas ni archivos locales.

## Experimento conceptual
¿Cómo cambian los pesos al activar una máscara causal? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.


### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Calcular atención explícitamente
Construimos Q/K/V y calculamos matriz de atención y salida en PyTorch.


In [ ]:
torch.manual_seed(SEED)
sequence = torch.randn(1, 6, 8, device=device)
projection_q, projection_k, projection_v = nn.Linear(8,8).to(device), nn.Linear(8,8).to(device), nn.Linear(8,8).to(device)
Q, K, V = projection_q(sequence), projection_k(sequence), projection_v(sequence)
scores = Q @ K.transpose(-2, -1) / np.sqrt(Q.shape[-1])
weights = torch.softmax(scores, dim=-1)
attended = weights @ V
print('Q/K/V:', Q.shape, K.shape, V.shape, '| salida:', attended.shape)
print('suma por fila:', weights.sum(dim=-1))


**Resultado e interpretación.** Cada fila de pesos suma uno y representa una distribución sobre posiciones consultadas. Sin entrenamiento, estos pesos ilustran el mecanismo, no una explicación aprendida.


### Aplicar máscara causal y visualizar
Enmascaramos posiciones futuras antes del softmax y comparamos mapas.


In [ ]:
causal_mask = torch.triu(torch.ones(6, 6, device=device, dtype=torch.bool), diagonal=1)
causal_scores = scores.masked_fill(causal_mask, float('-inf'))
causal_weights = torch.softmax(causal_scores, dim=-1)[0].detach().cpu().numpy()
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(weights[0].detach().cpu(), vmin=0, vmax=1, cmap='viridis'); axes[0].set_title('Atención completa')
axes[1].imshow(causal_weights, vmin=0, vmax=1, cmap='viridis'); axes[1].set_title('Máscara causal')
for ax in axes: ax.set(xlabel='Posición consultada', ylabel='Consulta')
plt.tight_layout(); plt.show()


**Resultado e interpretación.** La máscara triangular fuerza ceros hacia el futuro. La atención completa sirve para tareas bidireccionales; la causal es necesaria cuando no debe filtrarse información futura.


## Limitaciones y conclusiones clave
Los resultados dependen de esta semilla, tamaño de muestra, arquitectura y protocolo. No extrapoles métricas sintéticas a datos reales; separa decisiones de modelado de conclusiones generales. La celda de salida aporta la evidencia numérica y visual para responder la pregunta guía.

- Los parámetros y activaciones determinan cómo se representa la señal.
- La pérdida, el optimizador y el protocolo de evaluación forman parte del experimento.
- Una métrica aislada no describe todos los errores ni garantiza generalización.

## Ejercicios progresivos
1. **Guiado:** Guiado: verifica que todos los pesos por encima de la diagonal causal sean cero.
2. **Independiente:** Independiente: cambia `d_k` y observa cómo afecta la escala de logits sin la raíz.
3. **Desafío:** Desafío: explica por qué atención no impone por sí misma orden temporal y qué codificación posicional añadirías.

## Conexión con el siguiente laboratorio
Usa la representación, operación o criterio de evaluación de este laboratorio como punto de partida del siguiente paso de la secuencia.
